# 6. Comparison of Transformer Models (BERT, RoBERTa, XLNet)

This notebook implements a sentiment analysis task using three different transformer architectures and compares their performance on the same dataset.

In [ ]:
!pip install -q transformers datasets torch evaluate scikit-learn matplotlib seaborn

## 1. Setup and Dataset Loading

We will use the **IMDb Movie Reviews** dataset and take a subset for efficient training and comparison.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import classification_report, accuracy_score

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load IMDb dataset (small subset for comparison)
dataset = load_dataset("imdb")

small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

raw_datasets = DatasetDict({
    "train": small_train_dataset,
    "test": small_test_dataset
})

print(raw_datasets)

## 2. Model Definitions

We will compare:
1. **BERT** (bert-base-uncased)
2. **RoBERTa** (roberta-base)
3. **XLNet** (xlnet-base-cased)

In [ ]:
model_names = [
    "bert-base-uncased",
    "roberta-base",
    "xlnet-base-cased"
]

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

## 3. Training and Evaluation Function

We'll create a generic function to train and evaluate each model.

In [ ]:
results = {}

def train_and_eval(model_name):
    print(f"\n{'='*20} Training {model_name} {'='*20}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # RoBERTa and XLNet don't have pad_token set sometimes or need special handling
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

    tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"comparison_{model_name.split('/')[-1]}",
        evaluation_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        logging_steps=10,
        save_steps=1000,
        report_to="none"
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
    )

    # Train
    train_result = trainer.train()
    
    # Evaluate
    eval_result = trainer.evaluate()
    
    results[model_name] = {
        "accuracy": eval_result["eval_accuracy"],
        "eval_loss": eval_result["eval_loss"],
        "train_time": train_result.metrics["train_runtime"]
    }
    
    return results[model_name]

## 4. Run Comparisons

In [ ]:
for name in model_names:
    train_and_eval(name)

## 5. Summary and Comparison

In [ ]:
df_results = pd.DataFrame(results).T
print("\nOverall Performance Comparison:")
print(df_results)

plt.figure(figsize=(10, 6))

# Accuracy Plot
plt.subplot(1, 2, 1)
sns.barplot(x=df_results.index, y=df_results['accuracy'])
plt.title('Accuracy Comparison')
plt.xticks(rotation=45)

# Training Time Plot
plt.subplot(1, 2, 2)
sns.barplot(x=df_results.index, y=df_results['train_time'])
plt.title('Training Time Comparison (seconds)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 6. Analysis

Based on the results:
- **BERT**: Serves as the robust baseline.
- **RoBERTa**: Usually performs better than BERT due to better pre-training and more data.
- **XLNet**: Uses permutation language modeling which handles long-range dependencies better, but might be slower due to its autoregressive nature during training.